In [1]:
import pandas as pd
from src.recovery_model import RecoveryModel

pd.set_option("multi_sparse", False)
pd.set_option("display.float_format", "{:.2f}".format)

In [2]:
# Select a folder for the data to be used
folder = "test_2"  # choose between: test_1  test_2  Toy_WEEE_v2
layer_0 = "flow"
layer_1 = "product"
layer_2 = "component"
layer_3 = "material"
layer_4 = "element"

In [3]:
# This section insures that the structure of the excel files is consistent
# ! PLEASE IGNORE FOR NOW

layer_names = (layer_0, layer_1, layer_2, layer_3, layer_4)
all_symbols = {layer_1: "P*", layer_2: "C*", layer_3: "M*", layer_4: "E*"}

metadata = {
    "path": f"data/{folder}/",
    "filename": "metadata.csv",
}
composition = {
    "path": f"data/{folder}/",
    "filename": "composition.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Layer 1": layer_1,
        "Layer 2": layer_2,
        "Layer 3": layer_3,
        "Layer 4": layer_4,
        "Value": "data",
        "parameterCode": "parameterCode",
        "Year": "year",  # not considered at this stage
        "Scenario": "scenario",  # not considered at this stage
        "Location": "region",  # not considered at this stage
        "UoM": "unit",  # not considered at this stage
    },
    # only values accepted for the parameterCode column
    # ! do not change
    "parameterCode": {
        layer_2: "c-p",
        layer_3: "m-c",
        layer_4: "e-m",
    },
}

inputs = {
    "path": f"data/{folder}/",
    "filename": "inputs.csv",
    "mapper": {
        "Stock/Flow ID": layer_0,
        "Substance_main_parent": layer_1,
        "Value": "data",
        "Unit": "unit",
        "Year": "year",  # not considered at this stage
    },
}

tcs = {
    "path": f"data/{folder}/",
    "filename": "TCs.csv",
    "mapper": {
        # inflows / outflows
        "input_flow": "inflows",
        "output_flow": "outflows",
        # levels
        "input_layer": "input_layer1_level",
        "input_sub_layer": "input_layer2_level",
        "output_layer": "output_layer1_level",
        # keys
        "input_layer_key": "input_layer1_key",
        "input_sub_layer_key": "input_layer2_key",
        "output_target_key": "output_layer1_key",
        # other columns
        "value": "data",
        "process": "process",
        "Year": "year",  # not considered at this stage
    },
    "all_symbols": {
        layer_1: "P*",
        layer_2: "C*",
        layer_3: "M*",
        layer_4: "E*",
    },
}

---


# Recovery model


In [4]:
model = RecoveryModel(
    name=folder,
    metadata=metadata,
    composition=composition,
    inputs=inputs,
    tcs=tcs,
    layer_names=layer_names,
    save_intermediary_steps=False,
    save_duplicates=False,
)

---

# Metadata


In [5]:
model.dims

(8, 3, 4, 3, 3)

In [6]:
model.size

864

In [7]:
model.flows_eqs

,F1,F2,F3,F4,F5,F6,F7,F8
process,,,,,,,,
T1,1,-1,-1,0,0,1,0,0
T2,0,1,0,-1,-1,0,0,0
T3,0,0,0,0,1,0,1,-1
T4,0,0,1,1,0,-1,-1,0


---

# Model equation

$$(I - A^T)x = y$$

<img src="./doc/img/model_equation.png" alt="model_equation" width="350" />

In [8]:
model.lneqs  # the A matrix (it will be properly renamed later)

<864x864 sparse matrix of type '<class 'numpy.float64'>'
	with 441 stored elements in Compressed Sparse Row format>

In [9]:
model.y

<864x1 sparse array of type '<class 'numpy.int64'>'
	with 2 stored elements in Compressed Sparse Column format>

---

# Solver


In [10]:
model.solve(aggregate=False, pivot=False).fillna("")

,flow,product,component,material,element,data
0,F1,P1,,,,1000.00
1,F1,P1,C1,,,250.00
2,F1,P1,C1,M1,,130.00
3,F1,P1,C1,M1,E1,91.00
4,F1,P1,C1,M1,E2,39.00
...,...,...,...,...,...,...
175,F8,P1,C3,M2,E2,0.40
176,F8,P2,C1,M1,E1,0.23
177,F8,P2,C1,M2,E2,0.09
178,F8,P2,C2,M1,E1,0.28


In [11]:
model.solve(aggregate=False, pivot=True).fillna("")

flow,product,component,material,element,F1,F2,F3,F4,F5,F6,F7,F8
0,P1,,,,1000.00,,,,,,,
1,P1,C1,,,250.00,62.50,,,,,,
2,P1,C1,M1,,130.00,32.50,,,12.68,,,
3,P1,C1,M1,E1,91.00,22.75,,,8.87,,,1.60
4,P1,C1,M1,E2,39.00,9.75,,,3.80,,,
5,P1,C1,M2,,120.00,30.00,,,9.60,,,
6,P1,C1,M2,E1,117.60,29.40,,,9.41,,,
7,P1,C1,M2,E2,2.40,0.60,,,0.19,,,0.01
8,P1,C2,,,590.00,27.76,26.86,5.55,,11.24,,
9,P1,C2,M1,,389.40,18.32,17.73,3.66,1.83,7.42,2.08,


---

# Mass balance

In [12]:
# folder = "Toy_WEEE_v2"
mass_balance = pd.read_csv(f"consolidation/{folder}_solution_mass_balance.csv", index_col=0)
mass_balance.fillna("")

,product,component,material,element,process,F1,F2,F3,F4,F5,F6,F7,F8,mass_balance
0,P1,,,,T1,1000.00,,,,,,,,1000.00
1,P1,C1,,,T1,250.00,-62.50,,,,,,,187.50
2,P1,C1,M1,,T1,130.00,-32.50,,,0.00,,,,97.50
3,P1,C1,M1,E1,T1,91.00,-22.75,,,0.00,,,0.00,68.25
4,P1,C1,M1,E2,T1,39.00,-9.75,,,0.00,,,,29.25
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118,P2,C2,M1,E1,T3,0.00,0.00,0.00,0.00,0.97,0.00,0.76,-0.28,1.45
119,P2,C2,M1,E2,T3,0.00,0.00,0.00,0.00,1.14,0.00,0.89,,2.04
120,P2,C2,M2,,T3,0.00,0.00,0.00,0.00,1.31,0.00,0.36,,1.67
121,P2,C2,M2,E1,T3,0.00,0.00,0.00,0.00,0.65,0.00,0.18,,0.83


In [13]:
impossible_rows = mass_balance["mass_balance"] < 0
mass_balance[impossible_rows].fillna("")

,product,component,material,element,process,F1,F2,F3,F4,F5,F6,F7,F8,mass_balance
